In [1]:
!pip install Sastrawi
!pip install tensorflow
!pip freeze > requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 10.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import re
import string
import json
import pickle
import tensorflow as tf

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.utils import compute_class_weight
from tensorflow.keras.layers import (TextVectorization, Embedding, LSTM, Dense, Dropout)
from tensorflow.keras.utils import plot_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Model #untuk functional API
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

In [4]:
url = "https://raw.githubusercontent.com/Capstone-Catatanku/Data-Science/refs/heads/main/Data-clean/Data-clean.csv"
data = pd.read_csv(url)
data = data[data['deskripsi_transaksi'].str.split().str.len() >= 2].reset_index(drop=True)
print(f"Total data bersih: {len(data)}")
print()
print(data['kategori'].value_counts())



Total data bersih: 21268

kategori
Belanja           9531
Konsumsi          5886
Transportasi      1964
Tagihan            962
Lain-lain          882
Hiburan            808
Kesehatan          636
Tempat Tinggal     335
Investasi          178
Pendapatan          86
Name: count, dtype: int64


In [5]:
data.head()
data.info()
data['kategori'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21268 entries, 0 to 21267
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   tanggal              21268 non-null  object 
 1   deskripsi_transaksi  21268 non-null  object 
 2   kategori             21268 non-null  object 
 3   nominal              21268 non-null  float64
dtypes: float64(1), object(3)
memory usage: 664.8+ KB


,count
kategori,
Belanja,9531
Konsumsi,5886
Transportasi,1964
Tagihan,962
Lain-lain,882
Hiburan,808
Kesehatan,636
Tempat Tinggal,335
Investasi,178


**Preprocessing data**

In [6]:
def preprocessing(teks):
  teks = teks.lower()
  teks = re.sub(r'[^\w\s]', "", teks)
  teks = re.sub(r'\s+', ' ', teks)
  return teks.strip()

In [7]:
slang_dict = {
    "bli": "beli", "dapet": "dapat", "tdk": "tidak",
    "byr": "bayar", "gw": "saya", "dr": "dari", "yg": "yang",
    "dgn": "dengan", "utk": "untuk", "jd": "jadi", "krn": "karena",
    "udh": "sudah", "sdh": "sudah", "msh": "masih", "bs": "bisa",
    "nyicil": "cicilan", "cicil": "cicilan", "pesen": "pesan",
    "topup": "isi saldo", "kontrakan": "sewa"
}
def normalize_slang_words(teks):
    return ' '.join([slang_dict.get(w, w) for w in teks.split()])

def clean(teks):
  teks = preprocessing(teks)
  teks = normalize_slang_words(teks)
  return teks
data['deskripsi_transaksi'] = data['deskripsi_transaksi'].apply(clean)
data.head()

,tanggal,deskripsi_transaksi,kategori,nominal
0,2024-03-05,order daging dada ayam,Belanja,63000.0
1,2023-08-15,grabfood nasi goreng toping,Konsumsi,91000.0
2,2023-12-20,cicilan cosmos kipas angin,Belanja,292000.0
3,2024-07-23,beli charger charging casan adaptor,Belanja,79000.0
4,2023-01-19,checkout sweety store indonesiakacamata,Belanja,59000.0


In [8]:
X = data['deskripsi_transaksi'].values
y = data['kategori'].values
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
len(X_train), len(X_val), len(X_test)

(17014, 2127, 2127)

**TEXT VECTORIZATION**

In [9]:
# Initialize LabelEncoder
le = LabelEncoder()

# Fit on training labels and transform all sets
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)
y_test_encoded = le.transform(y_test)

In [10]:
max_tokens = 5000
seq_length = 20

vectorize_layer = TextVectorization(
    max_tokens= max_tokens,
    output_sequence_length=seq_length
)
vectorize_layer.adapt(X_train)

vocab = vectorize_layer.get_vocabulary()
with open('vocabulary.json', 'w', encoding='utf-8') as f:
  json.dump(vocab, f, ensure_ascii=False)

num_classes = len(le.classes_)
model = Sequential([
    vectorize_layer,
    Embedding(
        input_dim=max_tokens,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

#compile model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
print("Distribusi kategori (training set):")
import pandas as pd
kategori_counts = pd.Series(y_train).value_counts()
print(kategori_counts)
print()
print(f"Rasio terbanyak vs tersedikit: {kategori_counts.max() / kategori_counts.min():.1f}x")

Distribusi kategori (training set):
Belanja           7625
Konsumsi          4709
Transportasi      1571
Tagihan            769
Lain-lain          706
Hiburan            646
Kesehatan          509
Tempat Tinggal     268
Investasi          142
Pendapatan          69
Name: count, dtype: int64

Rasio terbanyak vs tersedikit: 110.5x


**class weight**

In [12]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)
class_weights = dict(enumerate(class_weights))
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [13]:
history = model.fit(X_train, y_train_encoded, epochs=20, callbacks=[early_stopping], class_weight=class_weights, validation_data=(X_val, y_val_encoded), batch_size=32)
print(history.history)
model.evaluate(X_test, y_test_encoded)

Epoch 1/20
532/532 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.8579 - loss: 0.9481 - val_accuracy: 0.9868 - val_loss: 0.0605
Epoch 2/20
532/532 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9827 - loss: 0.0726 - val_accuracy: 0.9873 - val_loss: 0.0497
Epoch 3/20
532/532 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9908 - loss: 0.0269 - val_accuracy: 0.9882 - val_loss: 0.0494
Epoch 4/20
532/532 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9945 - loss: 0.0106 - val_accuracy: 0.9882 - val_loss: 0.0414
Epoch 5/20
532/532 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9969 - loss: 0.0062 - val_accuracy: 0.9878 - val_loss: 0.0594
Epoch 6/20
532/532 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9967 - loss: 0.0047 - val_accuracy: 0.9911 - val_loss: 0.0455
Epoch 7/20
532/532 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9966 - loss: 0.0038 - val_accuracy: 0.9906 - val_loss: 0.0426
Epoch 8/20
532/532 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9975 - loss: 0.0032 - val_accuracy: 

[0.04354747384786606, 0.9873060584068298]

In [14]:
model.save('model_klasifikasi.keras')
print(" Model balanced tersimpan")

 Model balanced tersimpan


In [15]:
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print(" label_encoder.pkl tersimpan")

 label_encoder.pkl tersimpan


In [16]:
report = classification_report(y_test_encoded, model.predict(X_test).argmax(axis=1))
print(report)

67/67 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       953
           1       1.00      1.00      1.00        81
           2       1.00      1.00      1.00        18
           3       1.00      1.00      1.00        63
           4       0.99      0.98      0.98       588
           5       0.98      1.00      0.99        88
           6       1.00      1.00      1.00         9
           7       1.00      1.00      1.00        97
           8       0.97      1.00      0.99        34
           9       0.98      1.00      0.99       196

    accuracy                           0.99      2127
   macro avg       0.99      1.00      0.99      2127
weighted avg       0.99      0.99      0.99      2127

